In [ ]:
# Download dataset from local server
import os, subprocess
DATA_URL = "http://152.42.254.255:8765/captcha-data.zip"
subprocess.run(["wget", "-q", DATA_URL, "-O", "/tmp/captcha-data.zip"], check=True)
subprocess.run(["unzip", "-q", "-o", "/tmp/captcha-data.zip", "-d", "/tmp/captcha-training/"], check=True)
print(f'Dataset ready: {len(os.listdir("/tmp/captcha-training/raw"))} images')


In [ ]:
# Dataset + Augmentation + Model + Training + Upload - ALL IN ONE CELL
import os, json, random, string, subprocess
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
from torchvision import transforms, models

# === CONFIG ===
data_root = '/tmp/captcha-training'
labels_file = os.path.join(data_root, 'labeled/labels_clean.jsonl')
CHARSET = string.digits + string.ascii_uppercase + string.ascii_lowercase  # 62 chars, NO Cyrillic
CHAR_TO_IDX = {c: i+1 for i, c in enumerate(CHARSET)}
IDX_TO_CHAR = {i+1: c for i, c in enumerate(CHARSET)}
NUM_CLASSES = len(CHARSET) + 1
MAX_LEN = 6
EPOCHS = 60
BATCH_SIZE = 64
LR = 1e-4

def label_to_idx(label):
    return [CHAR_TO_IDX[c] for c in label if c in CHAR_TO_IDX]

def idx_to_text(indices):
    return ''.join(IDX_TO_CHAR[i] for i in indices if i > 0 and i in IDX_TO_CHAR)

# Load & filter samples
samples = [json.loads(l) for l in open(labels_file) if l.strip()]
valid = [s for s in samples if all(c in CHAR_TO_IDX for c in s.get('label','')) and len(s['label']) <= MAX_LEN]
print(f'Raw: {len(samples)} | Valid (clean charset): {len(valid)} | Skipped: {len(samples)-len(valid)}')
print(f'Charset ({len(CHARSET)}): {CHARSET}')

# Augmentation
class CaptchaAug:
    def __call__(self, img):
        if random.random() < 0.4: img = img.rotate(random.uniform(-10, 10), fillcolor=0)
        if random.random() < 0.3: img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.5, 1.2)))
        if random.random() < 0.3: img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.4))
        if random.random() < 0.3: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
        if random.random() < 0.2: img = ImageOps.posterize(img, bits=random.choice([4,5,6]))
        if random.random() < 0.3: img = img.filter(ImageFilter.SHARPEN)
        return img

train_tf = transforms.Compose([CaptchaAug(), transforms.Resize((64,256)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]), transforms.RandomErasing(p=0.15, scale=(0.02,0.08))])
val_tf = transforms.Compose([transforms.Resize((64,256)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class CaptchaDS(Dataset):
    def __init__(self, samples, tf): self.samples, self.tf = samples, tf
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        s = self.samples[i]
        img = Image.open(os.path.join(data_root, 'raw', s['file'])).convert('RGB')
        img = self.tf(img)
        li = label_to_idx(s['label'])
        return img, torch.tensor(li, dtype=torch.long), len(li)

def collate(b):
    imgs, labels, lens = zip(*b)
    return torch.stack(imgs), torch.cat(labels), torch.tensor(lens)

random.seed(42); random.shuffle(valid)
split = int(0.9 * len(valid))
train_dl = DataLoader(CaptchaDS(valid[:split], train_tf), BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate, pin_memory=True)
val_dl = DataLoader(CaptchaDS(valid[split:], val_tf), BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=collate, pin_memory=True)
print(f'Train: {split} | Val: {len(valid)-split}')

# Model
class CRNN(nn.Module):
    def __init__(self, nc):
        super().__init__()
        r = models.resnet18(weights=None)
        self.back = nn.Sequential(*list(r.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, 256, 2, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(512, nc)
    def forward(self, x):
        f = self.pool(self.back(x)).squeeze(2).permute(0,2,1)
        return self.fc(self.rnn(f)[0])

device = torch.device('cuda')
model = CRNN(NUM_CLASSES).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

# Train
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

def decode(preds):
    preds = preds.softmax(2).argmax(2).permute(1,0)
    res = []
    for p in preds:
        t, prev = '', -1
        for x in p:
            x = x.item()
            if x != 0 and x != prev and x in IDX_TO_CHAR: t += IDX_TO_CHAR[x]
            prev = x
        res.append(t)
    return res

def validate():
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels, lens in val_dl:
            imgs = imgs.to(device)
            decoded = decode(model(imgs))
            idx = 0
            for i, ll in enumerate(lens):
                target = idx_to_text(labels[idx:idx+ll].tolist())
                idx += ll
                if decoded[i] == target: correct += 1
                total += 1
    return correct / total if total else 0

best_acc = 0; no_imp = 0
print(f'Training {EPOCHS} epochs, lr={LR}, bs={BATCH_SIZE}')
print('-' * 60)

for ep in range(EPOCHS):
    model.train(); total_loss = 0
    for imgs, labels, lens in train_dl:
        imgs, labels, lens = imgs.to(device), labels.to(device), lens.to(device)
        preds = model(imgs)
        lp = preds.log_softmax(2).permute(1,0,2)
        il = torch.full((imgs.size(0),), preds.size(1), dtype=torch.long).to(device)
        loss = criterion(lp, labels, il, lens)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total_loss += loss.item()
    sched.step()
    avg = total_loss / len(train_dl)
    acc = validate()
    marker = ''
    if acc > best_acc:
        best_acc = acc; no_imp = 0; marker = ' *'
        torch.save({'model_state_dict': model.state_dict(), 'charset': CHARSET, 'num_classes': NUM_CLASSES, 'epoch': ep, 'val_acc': acc}, '/tmp/best_v2.pth')
    else:
        no_imp += 1
    if (ep+1) % 5 == 0 or acc == best_acc:
        print(f'Ep {ep+1:3d}/{EPOCHS} | Loss: {avg:.4f} | Val: {acc:.4f} | Best: {best_acc:.4f}{marker}')
    if no_imp >= 15:
        print(f'Early stop at epoch {ep+1}'); break

print(f'\nBest val accuracy: {best_acc:.4f}')

# Test predictions
ck = torch.load('/tmp/best_v2.pth')
model.load_state_dict(ck['model_state_dict'])
model.eval()
print(f'\nModel from epoch {ck["epoch"]+1}:')
with torch.no_grad():
    for i in range(min(10, len(valid[split:]))):
        s = valid[split:][i]
        img = val_tf(Image.open(os.path.join(data_root, 'raw', s['file'])).convert('RGB')).unsqueeze(0).to(device)
        pred = decode(model(img))[0]
        st = 'OK' if pred == s['label'] else 'MISS'
        print(f'{st} | True: {s["label"]:6s} | Pred: {pred:6s}')

# Upload to file.io
print('\nUploading to file.io...')
r = subprocess.run(['curl', '-s', '-F', 'file=@/tmp/best_v2.pth', 'https://file.io'], capture_output=True, text=True, timeout=120)
print(r.stdout[:500])
